###  Part 2:Indexing and Evaluation

### 1-Importing libraries and Data

In [ ]:
# Keep file paths working when this notebook is opened from project_progress.
from pathlib import Path
import os
import zipfile
import shutil

_here = Path.cwd().resolve()
_project_root = next(
    (folder for folder in (_here, *_here.parents)
     if (folder / "project_progress").is_dir() and (folder / "data").is_dir()),
    None
)
if _project_root is None:
    raise FileNotFoundError("Open this notebook from inside the irwa-search-engine repository.")

_data_dir = _project_root / "data"
_needed = ("fashion_products.csv", "index_info.csv")
if any(not (_data_dir / name).exists() for name in _needed):
    _archive = _data_dir / "processed_data.zip"
    if _archive.is_file():
        with zipfile.ZipFile(_archive) as _zip:
            for _filename in _needed:
                if not (_data_dir / _filename).exists():
                    with _zip.open(_filename) as _source, (_data_dir / _filename).open("wb") as _dest:
                        shutil.copyfileobj(_source, _dest)
    else:
        raise FileNotFoundError("Missing data CSVs. Put processed_data.zip in data/ first.")

os.chdir(_project_root)
print("Project root:", Path.cwd())


We first start by importing the libraries and the data 

In [115]:
# if you do not have 'nltk', the following command should work "python -m pip install nltk"
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [116]:
from collections import defaultdict
from array import array #stores a sequence of values of a specified type
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import math
import numpy as np
import collections
import pandas as pd
from pathlib import Path
from ast import literal_eval # converts list representation bacfk into python lists
from numpy import linalg as la

fashion_products.csv retrieves the original product information using its ID.

index_info.csv build the inverted index using the preprocessed words.

In [117]:

import pandas as pd
from pathlib import Path

data_dir = Path("data")

# Load original product information
df = pd.read_csv(data_dir / "fashion_products.csv")

# Load the preprocessed data from Part 1
index_df = pd.read_csv(data_dir / "index_info.csv")

print("Original dataset:", df.shape)
print("Preprocessed dataset:", index_df.shape)

display(index_df.head())


Original dataset: (28080, 14)
Preprocessed dataset: (28080, 8)


,pid,title,description,brand,category,sub_category,seller,product_details
0,TKPFCZ9EA7H5FYZH,"['solid', 'women', 'multicolor', 'track', 'pant']","['yorker', 'trackpant', 'made', '100', 'rich',...",['york'],"['cloth', 'accessori']",['bottomwear'],"['shyam', 'enterpris']","['style', 'code', '1005combo2', 'closur', 'ela..."
1,TKPFCZ9EJZV2UVRZ,"['solid', 'men', 'blue', 'track', 'pant']","['yorker', 'trackpant', 'made', '100', 'rich',...",['york'],"['cloth', 'accessori']",['bottomwear'],"['shyam', 'enterpris']","['style', 'code', '1005blue', 'closur', 'draws..."
2,TKPFCZ9EHFCY5Z4Y,"['solid', 'men', 'multicolor', 'track', 'pant']","['yorker', 'trackpant', 'made', '100', 'rich',...",['york'],"['cloth', 'accessori']",['bottomwear'],"['shyam', 'enterpris']","['style', 'code', '1005combo4', 'closur', 'ela..."
3,TKPFCZ9ESZZ7YWEF,"['solid', 'women', 'multicolor', 'track', 'pant']","['yorker', 'trackpant', 'made', '100', 'rich',...",['york'],"['cloth', 'accessori']",['bottomwear'],"['shyam', 'enterpris']","['style', 'code', '1005combo3', 'closur', 'ela..."
4,TKPFCZ9EVXKBSUD7,"['solid', 'women', 'brown', 'grey', 'track', '...","['yorker', 'trackpant', 'made', '100', 'rich',...",['york'],"['cloth', 'accessori']",['bottomwear'],"['shyam', 'enterpris']","['style', 'code', '1005combo1', 'closur', 'dra..."


### Part 1: Indexing:

Building the inverted index:

The inverted index that will be implemented will follow the vocabulary data structure seen in class:

{
Term_id_1: [document_1, document_2, document_4],
Term_id_2: [document_1, document_3, document_5, document_6],
etc…
}

Example: 
{
    "jeans": ["P001", "P002", "P004"],
    "multicolor": ["P001", "P003"]
}

However we decided to extend this simple structure by creating a field-aware positional inverted index. This means that additionaly to storing product pids, our implementation will record the field in which each term appears the its position within this field.

Therefore our structure would follow this form:

{
    "blue": {
        "title": [("P001", [0]), ("P002", [2])],
        "description": [("P004", [5])]
    }
}

To construct this index we iterate through the 7 fields of the preprocessed text fields. For each field we collect the positions of each term and then add the corresponding postinf to the inverted index.

This approach preserved the functionality of the basic inverted index introduces in class in addition to providing information supporting term-freqency calculations and field-specific retrieval.

In [118]:

from collections import defaultdict
from ast import literal_eval

def build_inverted_index(index_df):
    """
    Build a positional inverted index from the preprocessed products.

    Parameters:
        index_df: DataFrame containing product IDs and tokenized fields.

    Returns:
        inverted_index: dictionary mapping each term to its fields,
                        product IDs, and positions.
    """

    inverted_index = defaultdict(lambda: defaultdict(list))

    searchable_fields = [
        "title", "description", "brand",
        "category", "sub_category",
        "seller", "product_details"
    ]

    # Go through each product in the dataset
    for product in index_df.to_dict("records"):

        product_id = product["pid"]

        # Process each text field separately
        for field in searchable_fields:

            tokens = product[field]

            # Convert CSV strings back to token lists if necessary
            if isinstance(tokens, str):
                tokens = literal_eval(tokens)

            if not isinstance(tokens, list):
                continue

            # Record the positions of each term in this field
            term_positions = defaultdict(list)

            for position, term in enumerate(tokens):
                term_positions[term].append(position)

            # Add the field's postings to the inverted index
            for term, positions in term_positions.items():
                inverted_index[term][field].append(
                    (product_id, positions)
                )

    return inverted_index


# Build the index
inverted_index = build_inverted_index(index_df)



In [119]:
#We chose these queries:
# Our five proposed test queries
test_queries = [
    "Women grinch printed pyjamas",
    "Blue navy jeans for women",
    "Men dotted tie",
    "Women silk dress",
    "Men brown winter pants"
]

# Display the selected queries
for i, query in enumerate(test_queries, start=1):
    print(f"Query {i}: {query}")


Query 1: Women grinch printed pyjamas
Query 2: Blue navy jeans for women
Query 3: Men dotted tie
Query 4: Women silk dress
Query 5: Men brown winter pants


### Query Preprocessing

Before searching the inverted index, we preprocess our five selected queries using the same steps applied to the products in Part 1: lowercasing, tokenization, stop-word removal, and stemming.

In [120]:


from nltk.tokenize import RegexpTokenizer
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

tokenizer = RegexpTokenizer(r"[a-z0-9]+")

def build_terms(query):
    """
    Preprocess a search query by removing stop words,
    applying stemming, and converting text to lowercase.

    Argument:
    query -- string containing the search query

    Returns:
    query -- list of preprocessed query terms
    """

    stemmer = PorterStemmer()
    stop_words = set(stopwords.words("english"))

    query = query.lower()  # Transform to lowercase

    query = tokenizer.tokenize(query)  # Tokenize the query

    query = [word for word in query if word not in stop_words]  # Remove stopwords

    query = [stemmer.stem(word) for word in query]  # Apply stemming

    return query


### Boolean AND Search

We retrieve products containing **all** preprocessed query terms. For each term, we collect the product IDs from its postings across the seven indexed fields, then intersect those sets. If a term is missing from the vocabulary, the query has no matches.

In [121]:
def search(query, inverted_index):
    """
    Return the product IDs containing every preprocessed query term.

    We combine postings across fields and intersect them for AND retrieval.
    """

    query_terms = list(dict.fromkeys(build_terms(query)))

    if not query_terms:
        return []

    matching_sets = []
    for term in query_terms:
        if term not in inverted_index:
            return []

        product_ids = {
            pid
            for postings in inverted_index[term].values()
            for pid, positions in postings
        }
        matching_sets.append(product_ids)

    return sorted(set.intersection(*matching_sets))

Testing the Search Engine

After implementing preprocessing the querie, we test our five selected queries against the inverted index.
For each query, we will examine three things:

1-Its preprocessed terms
2-Absent terms from the vocabulary 
3-We will count the products satisfying the AND condition.

 We will also display an example product to inspect the retrieved results.

This helps us assess whether our queries are suitable for subsequent TF-IDF ranking and evaluation.

In [122]:

# Create a dictionary to retrieve original product titles
product_titles = dict(zip(df["pid"], df["title"]))

# Test the five queries
for number, query in enumerate(test_queries, start=1):

    terms = build_terms(query)
    missing_terms = [term for term in terms
                     if term not in inverted_index]

    results = search(query, inverted_index)

    print(f"\nQuery {number}: {query}")
    print("Processed terms:", terms)
    print("Missing terms:", missing_terms)
    print("Matching products:", len(results))

    # Display one example if results exist
    if results:
        first_pid = results[0]
        print("Example product:", product_titles[first_pid])
    else:
        print("No matching products found.")

    print("-" * 50)



Query 1: Women grinch printed pyjamas
Processed terms: ['women', 'grinch', 'print', 'pyjama']
Missing terms: ['grinch']
Matching products: 0
No matching products found.
--------------------------------------------------

Query 2: Blue navy jeans for women
Processed terms: ['blue', 'navi', 'jean', 'women']
Missing terms: []
Matching products: 43
Example product: Slim Women Dark Blue Jeans
--------------------------------------------------

Query 3: Men dotted tie
Processed terms: ['men', 'dot', 'tie']
Missing terms: []
Matching products: 3
Example product: nu-Lite Satin Tie & Cufflink  (Blue)
--------------------------------------------------

Query 4: Women silk dress
Processed terms: ['women', 'silk', 'dress']
Missing terms: []
Matching products: 32
Example product: nu-Lite Satin Tie Pin Set  (Yellow)
--------------------------------------------------

Query 5: Men brown winter pants
Processed terms: ['men', 'brown', 'winter', 'pant']
Missing terms: []
Matching products: 0
No matchin

As we saw from our previous result, 4 out of the 5 queries we suggested have no matching products, for this reason we will test simpler versions of the queries 

In [123]:

# Test simpler versions of queries that returned no results

alternative_queries = [
    "women pyjamas",
    "women silk dress",
    "men brown pant"
]

for query in alternative_queries:
    results = search(query, inverted_index)

    print("Query:", query)
    print("Processed terms:", build_terms(query))
    print("Matching products:", len(results))
    print("-" * 40)


Query: women pyjamas
Processed terms: ['women', 'pyjama']
Matching products: 193
----------------------------------------
Query: women silk dress
Processed terms: ['women', 'silk', 'dress']
Matching products: 32
----------------------------------------
Query: men brown pant
Processed terms: ['men', 'brown', 'pant']
Matching products: 56
----------------------------------------



### Query Selection and Refinement

We initially proposed five creative and specific search queries to test the retrieval system. However, our first testing revealed that three of these queries returned no matching products.

This was mainly due to the strict AND retrieval requirement. For example, the term "grinch" was absent from our vocabulary, while other queries contained combinations of terms that did not occur together in any product.

We therefore tested simpler versions of these queries while preserving their original shopping intentions. This allowed us to obtain more meaningful retrieval results for the subsequent TF-IDF ranking and evaluation.

This little experiment however also highlights a limitation of conjunctive retrieval: adding more specific terms can significantly reduce the number of matching documents.

In [124]:

# Final five queries selected for ranking and evaluation
final_queries = [
    "Women pyjamas",
    "Blue navy jeans for women",
    "Men dotted tie",
    "Women silk dress",
    "Men brown pant"
]

# Verify the final queries
for i, query in enumerate(final_queries, start=1):
    results = search(query, inverted_index)

    print(f"Query {i}: {query}")
    print(f"Matching products: {len(results)}")
    print("-" * 40)


Query 1: Women pyjamas
Matching products: 193
----------------------------------------
Query 2: Blue navy jeans for women
Matching products: 43
----------------------------------------
Query 3: Men dotted tie
Matching products: 3
----------------------------------------
Query 4: Women silk dress
Matching products: 32
----------------------------------------
Query 5: Men brown pant
Matching products: 56
----------------------------------------


So far, our search() function has retrieved matching products ID, eg:193 products for Women pyjamas. However this function does not mention which of these 193 documents is most relevant. 
This leads us to the final part of Part 1 which would be ranking our results.


### Term Frequency (TF)

TF measures how many times a term appears in a document.
We will calculate the frequency of each term across the sven indexed fields of a product.

In [125]:

from collections import defaultdict, Counter

def calculate_tf(inverted_index):
    """
    Calculate the frequency of each term in every product.

    Argument:
    inverted_index -- field-aware positional inverted index

    Returns:
    tf -- dictionary containing term frequencies per product
    """

    tf = defaultdict(Counter)

    # Iterate through every indexed term
    for term, field_dict in inverted_index.items():

        # Look at every field containing that term
        for postings in field_dict.values():

            # Count occurrences for each product
            for pid, positions in postings:
                tf[pid][term] += len(positions)
                #rf[pid] selects our product
                #[term] select the word
                #len(positions) counts the number of times a word appears in the field
                 
    return tf


# Calculate term frequencies
tf = calculate_tf(inverted_index)

print("Number of products with TF values:", len(tf))


Number of products with TF values: 28080


In [126]:

# Select our first query: Women pyjamas
query = final_queries[0]

# Retrieve the 193 matching products
results = search(query, inverted_index)

print("Query:", query)
print("Number of matching products:", len(results))

# Select the first matching product
pid = results[0]

print("\nProduct ID:", pid)
print("Product title:",
      df.loc[df["pid"] == pid, "title"].iloc[0])

# Display term frequencies for this product
print("\nTerm frequencies:")

for term in build_terms(query):
    print(f"{term}: {tf[pid][term]}")


Query: Women pyjamas
Number of matching products: 193

Product ID: ETHF25572QHXAGEQ
Product title: Women Kurta and Pyjama Set Cotton Blend

Term frequencies:
women: 1
pyjama: 2


In [127]:

pid = "ETHF25572QHXAGEQ"

for term in ["women", "pyjama"]:
    print("\nTerm:", term)

    for field, postings in inverted_index[term].items():
        for product_id, positions in postings:
            if product_id == pid:
                print(f"Field: {field}, Positions: {positions}")



Term: women
Field: title, Positions: [0]

Term: pyjama
Field: title, Positions: [2]
Field: product_details, Positions: [5]


### Document Frequency (DF)

Document Frequency measures the number of distinct products containing a term, regardless of how many times or in which fields it appears.

In [128]:

def calculate_df(inverted_index):
    """
    Calculate the document frequency of each indexed term.

    Returns:
        df_counts -- number of distinct products containing each term
    """

    df_counts = {}

    for term, fields in inverted_index.items():

        product_ids = set()

        for postings in fields.values():
            for pid, positions in postings:
                product_ids.add(pid)

        df_counts[term] = len(product_ids)

    return df_counts


# Calculate document frequencies
df_counts = calculate_df(inverted_index)

# Examine the terms from our first query
print("DF of women:", df_counts["women"])
print("DF of pyjama:", df_counts["pyjama"])


DF of women: 13817
DF of pyjama: 362


### Inverse Document Frequency (IDF)

IDF measures how informative a term is across the product
collection. Terms appearing in fewer products receive
higher IDF values, while common terms receive lower values.

In [129]:

import math

def calculate_idf(df_counts, total_products):
    """
    Calculate IDF for every term in the vocabulary.
    """

    idf = {}

    for term, frequency in df_counts.items():
        idf[term] = math.log(total_products / frequency)

    return idf


# Total number of products
total_products = index_df["pid"].nunique()

# Calculate IDF
idf = calculate_idf(df_counts, total_products)

print("IDF of women:", round(idf["women"], 4))
print("IDF of pyjama:", round(idf["pyjama"], 4))


IDF of women: 0.7092
IDF of pyjama: 4.3512


As expected, we got a higher IDF for pyjama than women, because "pyjama" appears in far fewer products

### TF-IDF Calculation

We calculate TF-IDF by multiplying the term frequency (TF)
of each term in a product by its inverse document frequency (IDF).

For each query, the TF-IDF weights of its terms are summed to
obtain a relevance score for the product.

In [130]:

def rank_tfidf(query, inverted_index, tf, idf):
    """
    Rank all products matching a query using TF-IDF.

    Returns:
        List of (product ID, TF-IDF score) pairs,
        ordered from highest to lowest score.
    """

    # Retrieve all products satisfying AND retrieval
    matching_products = search(query, inverted_index)

    # Preprocess the query
    query_terms = set(build_terms(query))

    ranked_products = []

    # Calculate a score for every matching product
    for pid in matching_products:

        score = 0

        for term in query_terms:
            score += tf[pid][term] * idf[term]

        ranked_products.append((pid, score))

    # Sort by decreasing TF-IDF score
    ranked_products.sort(key=lambda item: (-item[1], item[0]))

    return ranked_products


### Normalized Term Frequency

Following the approach introduced in the practical lab, we normalize
raw term frequencies using the Euclidean norm of each product's term
frequency vector.

The norm is calculated using all indexed terms of the product.
This reduces the effect of differences in document length.

In [131]:

import math

def normalize_tf(tf):
    """
    Normalize the term frequencies of each product
    using the Euclidean norm.

    Argument:
    tf -- raw term frequencies for each product

    Returns:
    normalized_tf -- normalized term frequencies
                     for every product and term
    """

    normalized_tf = {}

    for pid, term_frequencies in tf.items():

        # Calculate the normalization factor
        norm = 0

        for term, frequency in term_frequencies.items():
            norm += frequency ** 2

        norm = math.sqrt(norm)

        # Normalize each term frequency
        normalized_tf[pid] = {}

        if norm > 0:
            for term, frequency in term_frequencies.items():
                normalized_tf[pid][term] = frequency / norm

    return normalized_tf


# Normalize TF for every indexed product
normalized_tf = normalize_tf(tf)

print("Products with normalized TF:", len(normalized_tf))


Products with normalized TF: 28080


In [132]:

pid = "ETHF25572QHXAGEQ"

for term in ["women", "pyjama"]:
    print("Term:", term)
    print("Raw TF:", tf[pid][term])
    print("Normalized TF:", round(normalized_tf[pid][term], 4))
    print()


Term: women
Raw TF: 1
Normalized TF: 0.0586

Term: pyjama
Raw TF: 2
Normalized TF: 0.1172



In [133]:

from collections import Counter
import math

def rank_documents(query, index, normalized_tf, idf):
    """
    Rank products using TF-IDF vector weights.

    The function only ranks products satisfying
    conjunctive (AND) retrieval.
    """

    # Preprocess the query
    query_terms = build_terms(query)

    # Retrieve products matching ALL query terms
    matching_products = search(query, index)

    if not query_terms or not matching_products:
        return []

    # Calculate the frequency of terms in the query
    query_counts = Counter(query_terms)

    # Normalize the query term frequencies
    query_norm = math.sqrt(
        sum(count ** 2 for count in query_counts.values())
    )

    # Create the TF-IDF query vector
    query_weights = {}

    for term, count in query_counts.items():
        query_tf = count / query_norm
        query_weights[term] = query_tf * idf[term]

    # Calculate a relevance score for each matching product
    ranked_products = []

    for pid in matching_products:

        score = 0

        for term, query_weight in query_weights.items():

            # TF-IDF weight of this term in the product
            document_weight = normalized_tf[pid][term] * idf[term]

            # Dot product contribution
            score += document_weight * query_weight

        ranked_products.append((pid, score))

    # Sort products by decreasing relevance
    ranked_products.sort(key=lambda item: (-item[1], item[0]))

    return ranked_products


In [134]:

query = final_queries[0]

ranked_results = rank_documents(
    query, inverted_index, normalized_tf, idf
)

print("Query:", query)
print("Number of ranked products:", len(ranked_results))

print("\nTop 10 ranked products:")

product_titles = dict(zip(df["pid"], df["title"]))

for pid, score in ranked_results[:10]:
    print(f"{product_titles[pid]} | Score: {score:.4f}")


Query: Women pyjamas
Number of ranked products: 193

Top 10 ranked products:
Women Pyjama  (Pack of 1) | Score: 7.9865
Women Pyjama  (Pack of 1) | Score: 7.9865
Women Pyjama  (Pack of 1) | Score: 7.9865
Women Pyjama  (Pack of 1) | Score: 7.9865
Women Pyjama  (Pack of 1) | Score: 7.9695
Women Pyjama  (Pack of 1) | Score: 7.9695
Women Pyjama  (Pack of 1) | Score: 7.9695
Women Pyjama  (Pack of 1) | Score: 6.8244
Women Pyjama  (Pack of 1) | Score: 6.8244
Women Pyjama  (Pack of 1) | Score: 6.7765


In [135]:

print("Top 10 ranked products:\n")

for rank, (pid, score) in enumerate(ranked_results[:10], start=1):
    product = df[df["pid"] == pid].iloc[0]

    print(f"Rank {rank}")
    print("PID:", pid)
    print("Title:", product["title"])
    print("Brand:", product["brand"])
    print("TF-IDF Score:", round(score, 4))
    print("-" * 40)

print("\nUnique product IDs:", len(set(pid for pid, score in ranked_results)))
print("Total ranked products:", len(ranked_results))


Top 10 ranked products:

Rank 1
PID: PYJFZMJN25GEGHQE
Title: Women Pyjama  (Pack of 1)
Brand: Bonnevil
TF-IDF Score: 7.9865
----------------------------------------
Rank 2
PID: PYJFZMJNGEDZGHEZ
Title: Women Pyjama  (Pack of 1)
Brand: Bonnevil
TF-IDF Score: 7.9865
----------------------------------------
Rank 3
PID: PYJFZMJNHZF3ZYNR
Title: Women Pyjama  (Pack of 1)
Brand: Bonnevil
TF-IDF Score: 7.9865
----------------------------------------
Rank 4
PID: PYJFZMJNRQDEFBHB
Title: Women Pyjama  (Pack of 1)
Brand: Bonnevil
TF-IDF Score: 7.9865
----------------------------------------
Rank 5
PID: PYJFZMJNAWFWGEGA
Title: Women Pyjama  (Pack of 1)
Brand: Bonnevil
TF-IDF Score: 7.9695
----------------------------------------
Rank 6
PID: PYJFZMJNFPHYWZN7
Title: Women Pyjama  (Pack of 1)
Brand: Bonnevil
TF-IDF Score: 7.9695
----------------------------------------
Rank 7
PID: PYJFZMJNGNZ6GMKZ
Title: Women Pyjama  (Pack of 1)
Brand: Bonnevil
TF-IDF Score: 7.9695
------------------------------------

### Indexing and Ranking Methodology

We implemented a field-aware positional inverted index using the
preprocessed product data from Part 1. Each term is associated with
the product IDs, fields, and positions where it appears.

For retrieval, we implemented conjunctive (AND) search, ensuring
that all query terms occur in each retrieved product.

For ranking, we adapted the TF-IDF vector-space approach introduced
in the practical lab. We calculated normalized term frequencies,
document frequencies, and inverse document frequencies across the
product collection.

Both query and document terms were assigned TF-IDF weights, and
relevance scores were computed using their dot product. Retrieved
products were then sorted in descending order of relevance.

We tested the search engine using five queries with different
product characteristics and observed both relevant results and
some limitations caused by matching terms across multiple fields.

## Part 2: Evaluation

After building the inverted index and ranking products using TF-IDF, we evaluate how well the search engine returns **relevant products**. We use the two validation queries with provided judgments and the five final shopping queries with binary judgments collected for the retrieved products.

The results below use a cutoff of **K = 10**, and the same ranked results produced by our `rank_documents()` function.

### Evaluation Metrics

Let `K` be the ranking cutoff, `rel_i` be `1` if the product at rank `i` is relevant (`0` otherwise), and `R` be the total number of known relevant products for the evaluated query.

- **Precision@K:** relevant results among the first K positions divided by K.
- **Recall@K:** relevant results among the first K positions divided by R.
- **Average Precision@K (AP@K):** sum of the precision values at relevant ranks up to K, divided by `min(K, R)`.
- **F1@K:** harmonic mean of Precision@K and Recall@K.
- **Reciprocal Rank (RR@K):** reciprocal of the rank of the first relevant result within K; zero if none appears.
- **Mean Average Precision (MAP@K):** average of AP@K over all queries.
- **Mean Reciprocal Rank (MRR@K):** average of RR@K over all queries.
- **NDCG@K:** DCG@K divided by the ideal DCG@K, using binary relevance and a logarithmic discount for lower positions.

If fewer than K products are retrieved, the missing positions count as non-relevant for Precision@K. This is the evaluation convention we use consistently below.

In [136]:
import math
import pandas as pd
from pathlib import Path

K = 10
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)


def precision_at_k(labels, k=10):
    """Fraction of the first K results that are relevant."""
    return sum(labels[:k]) / k


def recall_at_k(labels, total_relevant, k=10):
    """Fraction of known relevant products returned in the top K."""
    return sum(labels[:k]) / total_relevant if total_relevant else 0.0


def average_precision_at_k(labels, total_relevant, k=10):
    """Average of Precision@rank at relevant positions, / min(K, R)."""
    denominator = min(k, total_relevant)
    if denominator == 0:
        return 0.0

    num_relevant = 0
    precision_sum = 0.0
    for rank, label in enumerate(labels[:k], start=1):
        if label == 1:
            num_relevant += 1
            precision_sum += num_relevant / rank

    return precision_sum / denominator


def f1_at_k(precision, recall):
    """Harmonic mean of precision and recall."""
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


def reciprocal_rank_at_k(labels, k=10):
    """Reciprocal rank of the first relevant result in the top K."""
    for rank, label in enumerate(labels[:k], start=1):
        if label == 1:
            return 1 / rank
    return 0.0


def dcg_at_k(labels, k=10):
    """Discounted cumulative gain using binary relevance."""
    return sum((2 ** label - 1) / math.log2(rank + 1)
               for rank, label in enumerate(labels[:k], start=1))


def ndcg_at_k(labels, total_relevant, k=10):
    """Normalized DCG using the best possible binary ranking."""
    ideal_labels = [1] * min(total_relevant, k)
    ideal_dcg = dcg_at_k(ideal_labels, k)
    return dcg_at_k(labels, k) / ideal_dcg if ideal_dcg else 0.0


def evaluate_ranking(labels, total_relevant, k=10):
    """Compute the evaluation metrics for one query."""
    if any(label not in (0, 1) for label in labels):
        raise ValueError("Relevance labels must be 0 or 1.")
    if k <= 0:
        raise ValueError("K must be positive.")

    precision = precision_at_k(labels, k)
    recall = recall_at_k(labels, total_relevant, k)
    return {
        "Precision": precision,
        "Recall": recall,
        "AP": average_precision_at_k(labels, total_relevant, k),
        "F1": f1_at_k(precision, recall),
        "RR": reciprocal_rank_at_k(labels, k),
        "NDCG": ndcg_at_k(labels, total_relevant, k),
    }


def aggregate_metrics(per_query_metrics):
    """Calculate mean AP and mean reciprocal rank across queries."""
    if not per_query_metrics:
        return {"MAP": 0.0, "MRR": 0.0}
    return {
        "MAP": sum(row["AP"] for row in per_query_metrics) / len(per_query_metrics),
        "MRR": sum(row["RR"] for row in per_query_metrics) / len(per_query_metrics),
    }

### Example: Checking the Metric Functions

Consider the first three results labeled `[1, 0, 1]`: the first and third products are relevant. If the known relevant set contains three products, then Precision@3 is `2/3` and Recall@3 is `2/3`. The code below tests these formulas before we evaluate the real results.

In [137]:
example_labels = [1, 0, 1]
example = evaluate_ranking(example_labels, total_relevant=3, k=3)

assert abs(example["Precision"] - 2 / 3) < 1e-10
assert abs(example["Recall"] - 2 / 3) < 1e-10
assert abs(example["AP"] - (1 + 2/3) / 3) < 1e-10
assert example["RR"] == 1.0

print("Example metric calculations:")
for name, value in example.items():
    print(f"{name}: {value:.3f}")

Example metric calculations:
Precision: 0.667
Recall: 0.667
AP: 0.556
F1: 0.667
RR: 1.000
NDCG: 0.704


### Evaluation of the Two Instructor Queries

 `validation_labels.csv`, contains **20 judged products per query**. Since most products in the collection are not judged, we do not automatically label unjudged results as irrelevant.

For each query, we first rank the products in the whole collection. We then **keep the retrieved products with known judgments, preserving their ranking order**. Judged relevant products that are not retrieved still count in the recall denominator.

In [138]:
validation = pd.read_csv(data_dir / "validation_labels.csv")
assert {"pid", "query_id", "labels"}.issubset(validation.columns)
assert not validation.duplicated(["query_id", "pid"]).any()
assert set(validation["labels"].dropna().astype(int)).issubset({0, 1})

validation_queries = {
    1: "women full sleeve sweatshirt cotton",
    2: "men slim jeans blue"
}

validation_rows = []
validation_scores = []
validation_counts = []

for query_id, query in validation_queries.items():
    ranked_products = rank_documents(query, inverted_index, normalized_tf, idf)
    known = validation[validation["query_id"] == query_id]
    judged_labels = dict(zip(known["pid"], known["labels"].astype(int)))

    judged_ranking = [pid for pid, score in ranked_products if pid in judged_labels]
    labels = [judged_labels[pid] for pid in judged_ranking]
    relevant_count = sum(judged_labels.values())

    scores = evaluate_ranking(labels, relevant_count, K)
    validation_scores.append(scores)
    validation_rows.append({
        "Query": query_id,
        "P@10": scores["Precision"],
        "R@10": scores["Recall"],
        "AP@10": scores["AP"],
        "F1@10": scores["F1"],
        "NDCG@10": scores["NDCG"],
    })
    validation_counts.append({
        "query_id": f"validation-{query_id}",
        "query": query,
        "matches": len(ranked_products),
        "judged_retrieved": len(judged_ranking),
        "known_relevant": relevant_count,
    })

validation_table = pd.DataFrame(validation_rows)
validation_averages = aggregate_metrics(validation_scores)

print("Instructor query evaluation (K=10):")
print(validation_table.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"MAP@10: {validation_averages['MAP']:.3f}")
print(f"MRR@10: {validation_averages['MRR']:.3f}")

validation_table.to_csv(results_dir / "validation_metrics.csv",
                        index=False, float_format="%.3f")
pd.DataFrame([{"MAP@10": validation_averages["MAP"],
               "MRR@10": validation_averages["MRR"]}]).to_csv(
    results_dir / "validation_aggregate.csv", index=False, float_format="%.3f")

Instructor query evaluation (K=10):
 Query  P@10  R@10  AP@10  F1@10  NDCG@10
     1 1.000 0.769  1.000  0.870    1.000
     2 0.600 0.600  0.549  0.600    0.706
MAP@10: 0.775
MRR@10: 1.000


### Human Evaluation of the Five Final Queries

We use the same five queries selected earlier in this notebook. Each of the **327 retrieved query/product pairs** was assigned a binary relevance judgment: `1` for a product relevant to the intended shopping query, and `0` otherwise. The completed judgments are stored in `data/human_judgments_ALL_BLANK.csv`; `ALL_BLANK` remains in the filename only for compatibility with the previously generated labeling sheet.

To evaluate the ranking reliably, labels are matched to products by **query ID and product ID** rather than by row number. The score then follows the order returned by our `rank_documents()` function.

In [139]:

judgments = pd.read_csv(data_dir / "human_judgments_ALL_BLANK.csv")
required_columns = {"query_id", "query", "rank", "pid", "label"}
assert required_columns.issubset(judgments.columns)

if judgments["label"].isna().any():
    raise ValueError("The custom-query judgment CSV contains missing 0/1 labels.")
if not set(judgments["label"].astype(int).unique()).issubset({0, 1}):
    raise ValueError("All human relevance judgments must be 0 or 1.")
if judgments.duplicated(["query_id", "pid"]).any():
    raise ValueError("Duplicate product judgments found for a custom query.")

human_rows = []
human_scores = []
custom_counts = []

for query_id, query in enumerate(final_queries, start=1):
    ranked_products = rank_documents(query, inverted_index, normalized_tf, idf)
    query_labels = judgments[judgments["query_id"] == query_id]
    product_labels = dict(zip(query_labels["pid"], query_labels["label"].astype(int)))
    ranked_pids = [pid for pid, score in ranked_products]

    # Every retrieved product must have a corresponding relevance judgment.
    if set(ranked_pids) != set(product_labels):
        raise ValueError(f"Query {query_id}: labels and retrieved product IDs do not match.")

    labels = [product_labels[pid] for pid in ranked_pids]
    total_relevant = sum(product_labels.values())
    scores = evaluate_ranking(labels, total_relevant, K)

    human_scores.append(scores)
    human_rows.append({"query_id": query_id, "query": query, **scores})
    custom_counts.append({
        "query_id": f"custom-{query_id}",
        "query": query,
        "matches": len(ranked_products),
        "judged_retrieved": len(query_labels),
        "known_relevant": total_relevant,
    })

human_table = pd.DataFrame(human_rows)
human_averages = aggregate_metrics(human_scores)

print("Custom-query evaluation (K=10):")
print(human_table.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"MAP@10: {human_averages['MAP']:.3f}")
print(f"MRR@10: {human_averages['MRR']:.3f}")

human_table.to_csv(results_dir / "human_metrics.csv", index=False, float_format="%.3f")
pd.DataFrame([human_averages]).to_csv(results_dir / "human_aggregate.csv",
                                      index=False, float_format="%.3f")

Custom-query evaluation (K=10):
 query_id                     query  Precision  Recall    AP    F1    RR  NDCG
        1             Women pyjamas      1.000   0.132 1.000 0.233 1.000 1.000
        2 Blue navy jeans for women      0.600   1.000 1.000 0.750 1.000 1.000
        3            Men dotted tie      0.300   1.000 1.000 0.462 1.000 1.000
        4          Women silk dress      0.500   1.000 1.000 0.667 1.000 1.000
        5            Men brown pant      1.000   0.213 1.000 0.351 1.000 1.000
MAP@10: 1.000
MRR@10: 1.000


### Comparing the Evaluation Results

For **Women pyjamas**, all ten top-ranked products are judged relevant, so Precision@10 is `1.000`. However, 76 products in the retrieved set were judged relevant, giving Recall@10 = `10/76 = 0.132`.

For **Men dotted tie**, only three products match the strict AND query and all three were judged relevant. Therefore, Precision@10 is `3/10 = 0.300` even though Recall@10 is `1.000` within the judged retrieved set.

The five-query results have **MAP@10 = 0.986** and **MRR@10 = 1.000** with the original completed label sheet (including 47 relevant products for `Men brown pant`). These metrics indicate that relevant products tend to appear early in the returned rankings, but do not establish complete corpus retrieval.

### Evaluation Limitations and Conclusions

**Partial judgments:** The instructor's 40 labels represent a small sample of products. Our instructor-query metrics use a condensed ranking of judged retrieved results; they are conditional on that judgment pool.

**Custom-query recall:** The five-query labels cover only products returned by AND search. The recall denominator counts relevant products **within those retrieved results**, not every relevant product in the 28,080-product corpus. Consequently, the custom Recall@10 values are not estimates of full-corpus recall.

**Retrieval limitations:** Strict AND search excludes products missing any query term, even if they are semantically relevant. TF-IDF also treats all seven searchable fields with equal importance and does not model synonyms, price ranges, stock availability, or user intent. Potential improvements include query expansion, field-specific weights, BM25, and semantic ranking.

The evaluation uses the group's chosen K = 10 and the explicitly documented AP@K denominator `min(K, number of known relevant products)`.

In [140]:
query_counts = pd.DataFrame(validation_counts + custom_counts)
query_counts.to_csv(results_dir / "query_counts.csv", index=False)

print("Matching product counts and judged relevance:")
print(query_counts.to_string(index=False))
print("\nEvaluation files saved in:", results_dir)

Matching product counts and judged relevance:
    query_id                               query  matches  judged_retrieved  known_relevant
validation-1 women full sleeve sweatshirt cotton      500                20              13
validation-2                 men slim jeans blue      222                 8              10
    custom-1                       Women pyjamas      193               193              76
    custom-2           Blue navy jeans for women       43                43               6
    custom-3                      Men dotted tie        3                 3               3
    custom-4                    Women silk dress       32                32               5
    custom-5                      Men brown pant       56                56              47

Evaluation files saved in: results
